In [ ]:
import pyterrier as pt
from pyterrier_dr import TctColBert, FlexIndex
from pyterrier_adaptive import GAR
from typing import Optional
import numpy as np
from collections import Counter
import pyterrier as pt
import pandas as pd
import ir_datasets
# Typing imports for type annotations
from pyterrier.model import add_ranks
import torch
from torch.nn import functional as F
from transformers import T5Config, T5Tokenizer, T5ForConditionalGeneration
from pyterrier.transformer import TransformerBase
import re
import scipy.sparse
import torch
import scipy
from pyterrier_t5 import MonoT5ReRanker
import json
from pyterrier_pisa import PisaIndex
from src.utils import *
from src.lightningmodule import *
from src.GNN import *
import warnings
import copy

logger = ir_datasets.log.easy()
if not pt.started(): 
    pt.init()
set_determinism_the_old_way(deterministic = True)


In [ ]:
dataset_id = "msmarco_data"
embedding_name = 'tctcolbert'
K = 8
flex_index = FlexIndex(index_path=f'./data/msmarco-index_{embedding_name}/')

# Generate the corpus graph using the corpus_graph method of FlexIndex.
graph = flex_index.corpus_graph(k=K)

In [ ]:
# bm25 = PisaIndex.from_dataset('msmarco_passage').bm25()
bm25 =  pt.BatchRetrieve.from_dataset('msmarco_passage', 'terrier_stemmed', wmodel='BM25')
text_field = "text"

# monoT5 = MonoT5ReRanker(text_field=text_field)
TCTC = TctColBert('castorini/tct_colbert-msmarco')
# TCTC2 = TctColBert('castorini/tct_colbert-v2-hnp-msmarco')



In [ ]:

class GNRR_Scorer(TransformerBase):
    def __init__(self,
                 batch_size=16,
                 text_field='text',
                 fast = False,
                 qrels = None,
                 config = None,
                 verbose=True, get_data = False, dataset = None):
        self.verbose = verbose
        self.batch_size = batch_size
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_name = 'castorini/tct_colbert-msmarco'
        self.encoder = TctColBert(self.model_name, device = self.device)
        self.text_field = text_field
        n = 768
        self.fast = fast
        self.get_data = get_data
        input_features = n if config.aggr != 'concat' else n * 2
        self.dataset_n = dataset
        self.qrels = qrels

        if config.modality == 'local' or config.modality == 'multistage':
            mod = GNN_LG(input_features, config, modality = config.modality, conv_type = config.conv_type, device = self.device)

        elif config.modality == 'global':
            print("Work in progress...")
        elif config.modality == 'single':

            if config.conv_type != 'mlp':
                mod = GNN_NR(input_features, config, output_dim = 1, device=self.device)
            else:
                mod = MLP(input_features, config.hidden_dim, output_dim = 1, n_layers=config.n_layers, device=self.device, dropout_prob=config.dropout_prob)
        
        
        if config.model_path != '':
            model_checkpoint = torch.load(f'{config.model_path}', map_location=self.device)
            mod.load_state_dict(model_checkpoint, strict=False)
        
        self.model = mod
        self.config = config

    def __str__(self):
        return f"GNRR_Scorer({self.model_name})"

    def transform(self, run, corpus_graph, corpus_graph_payload = None):
        self.model.eval()
        with torch.no_grad():    
        # CHANGE
            if self.get_data:
                dataset_n = self.dataset_n
                q_id = run['qid'].iloc[0]
                print("QID: ", q_id)
                path = f'./data/msmarco_data/val_data_fast/tensors/qid_{str(q_id)}_tensors/'
                if os.path.exists(path):
                   print("Path exists")
                   return

            topk_documents_df = run.drop_duplicates(subset='docno')

            if len(topk_documents_df) < 1000:
                print("Less than 1000")
            
            
                            
            if self.get_data:
                
                # Retrieve the relevance labels for the top-k documents
                merged_df = topk_documents_df.merge(self.qrels.loc[:, ['qid', 'docno', 'label']], on=['qid', 'docno'], how='left')
                merged_df = merged_df.fillna(-1)
            
            queries, texts = topk_documents_df['query'], topk_documents_df[self.text_field]

            docs = texts
            # print(query)
            query_enc = self.encoder.encode_queries(queries.iloc[0:1])[0]

            docno_to_index = {docno: idx for idx, docno in enumerate(topk_documents_df['docno'].unique())}


        
            index_to_docno = {docno_to_index[k]: k for k in docno_to_index}

            
            if not self.fast:
                doc_encs = self.encoder.encode_docs(docs)
            else:
                doc_encs = np.empty((len(topk_documents_df), query_enc.shape[0]), dtype=query_enc.dtype)
                
                try:
                    print("not SKIPPED")

                    for doc in range(len(topk_documents_df)):
                        docno = index_to_docno[doc]
                        real_id = corpus_graph_payload[0][docno]
                        doc_encs[doc] = corpus_graph_payload[1][real_id]
                except IndexError:
                    print("SKIPPED")
                    doc_encs = self.encoder.encode_docs(docs)

                

            if self.get_data:
                dataset_n = self.dataset_n
                q_id = run['qid'].iloc[0]

                print("QID: ", q_id)
                
                path = f'./data/{dataset_n}/val_data_fast/tensors/qid_{str(q_id)}_tensors/'

                os.makedirs(path, exist_ok=True)

                rels = torch.from_numpy(np.array(merged_df['label']))

                print("Rels have shape: ", rels.shape)
                torch.save(rels, f'{path}qrels_tensor_new.pt')

                path2 = f'./data/{dataset_n}/val_data_fast/metrics_directory/'#
                os.makedirs(path2, exist_ok=True)
                with open(path2+f'{str(q_id)}.json', 'w') as f:
                    json.dump(index_to_docno, f)
                

            corpus_sb = generate_corpus_subgraph_induced_by_query(topk_documents_df = topk_documents_df, complete_corpus_graph = corpus_graph)


            assert len(docno_to_index) == len(index_to_docno)

            for idx in range(len(doc_encs)):
                assert index_to_docno[idx] == get_key_from_value(docno_to_index, idx)

            adj_matrix = build_adjacency_matrix(corpus_sb, docno_to_index)

            adj_matrix = adjacency_matrix_to_coo(adj_matrix)
            
            query_feat = (torch.from_numpy(query_enc).clone().unsqueeze(0))
            
            x = (torch.from_numpy(doc_encs).clone()) 
            
            if self.get_data:
                torch.save(adj_matrix, f'{path}adjacency_matrix.pt')
                
                print("Adj have shape: ", adj_matrix.shape)
            
                torch.save(query_feat, f'{path}query_tensor_new.pt')
                print("query_feat have shape: ", query_feat.shape)
                torch.save(x, f'{path}doc_feat_tensor_new.pt')
                print("Doc have shape: ", x.shape)
            
            if len(topk_documents_df) <= 1:
                return 
            
            query_feat = query_feat.unsqueeze(0).to(self.device)

            x = x.unsqueeze(0).to(self.device)
          
            A = adj_matrix.unsqueeze(0).to(self.device)

            out = compute_output(x, A, query_feat, self.model, self.config.aggr, self.config.conv_type)

        ordered_list = sorted(list(index_to_docno.keys()))
        topk_indices = torch.topk(out, k = out.shape[0]).indices
        data = {
            'qid': [str(run['qid'].iloc[0])]*len(ordered_list),
            'query': queries,
            'docno': [],
            'score': [], 
            'rank': []
        }

        for doc in ordered_list:

            data['docno'].append(index_to_docno[doc])
            data['score'].append(out[doc].item())
            data['rank'].append(topk_indices.tolist().index(doc))
            
        df = pd.DataFrame(data)
        df = df.sort_values(by='rank')
    
        
        return df
    
def generate_corpus_subgraph_induced_by_query(                                          
    topk_documents_df: pd.DataFrame,
    complete_corpus_graph
) -> Dict[str, List[str]]:
    """
    Constructs and refines a corpus subgraph, focusing on relationships within a document subset.

    Conceptual Steps:
    1. Define Nodes: Identify and set the documents of interest as nodes in our subgraph. This step
    uses the 'documents_df' to extract document numbers, which will serve as nodes.

    2. Draw Edges: For each node, retrieve potential connections (edges) from the complete corpus graph.
    This involves fetching neighbors for each document from the comprehensive graph structure.

    3. Filter Edges: Refine the connections by ensuring each node (document) only connects to other nodes
    (documents) within our subset. This filtering process removes edges that lead outside the
    specified subset, maintaining the subgraph's integrity.

    4. Construct Subgraph: Populate the subgraph with nodes and their valid, filtered connections. This
    results in a dictionary where each key is a document number, and its value is a list of neighbor
    document numbers—all within the subset (i.e., valid neighbours).

    Args:
    - documents_df (pd.DataFrame): DataFrame containing documents of interest, identified by 'docno'.
    - graph_reference (NpTopKCorpusGraph): The complete corpus graph for neighbor retrieval.

    Returns:
    - Dict[str, List[str]]: Represents the corpus subgraph. Keys are document numbers ('docno'),
    and values are lists of neighbor document numbers, ensuring all are within the specified subset.
    """

    # Step 1: Define Nodes
    # Extract a set of document numbers to serve as valid nodes within our subgraph.
    valid_docnos = set(topk_documents_df['docno'])

    # Initialize the subgraph
    corpus_subgraph = {}
    found = 0
    for docno in valid_docnos:
        # Step 2: Draw Edges
        # Retrieve neighbors for the current document from the complete corpus graph.
        # CHANGE
        try:
            all_neighbors = complete_corpus_graph.neighbours(docno)
            found += 1
        except LookupError:
            warnings.warn(f"Document {docno} not found in the corpus graph.")
            continue
        # Step 3: Filter Edges
        # Filter these neighbors to include only those also present in our subset (valid_docnos).
        valid_neighbors = [neighbor for neighbor in all_neighbors if neighbor in valid_docnos]

        # Step 4: Construct Subgraph
        # Update our subgraph to include the current document and its filtered neighbors.
        corpus_subgraph[docno] = valid_neighbors  # Populate subgraph
    
    return corpus_subgraph
    
def build_adjacency_matrix(subgraph: Dict[str, list], docno_to_index: Dict[str, int]) -> np.ndarray:
    """
    Generates an adjacency matrix from a subgraph and a mapping of document numbers to indices.

    Parameters:
    subgraph (Dict[str, list]): A dictionary representing the subgraph with document numbers as keys.
    docno_to_index (Dict[str, int]): A dictionary mapping document numbers to their respective indices.

    Returns:
    np.ndarray: A symmetric adjacency matrix representing the graph.
    """
    # Error handling: Check if inputs are dictionaries
    if not isinstance(subgraph, dict) or not isinstance(docno_to_index, dict):
        raise ValueError("Both subgraph and docno_to_index must be dictionaries.")

    # Determine the size of the adjacency matrix
    # CHANGE
    matrix_size = len(docno_to_index)
    
    adjacency_matrix = np.zeros((matrix_size, matrix_size), dtype=int)

    # Iterate over each document and its neighbors in the subgraph
    for doc, neighbors in subgraph.items():
        if doc not in docno_to_index:
            raise KeyError(f"Document number {doc} not found in docno_to_index mapping.")
        doc_index = docno_to_index[doc]

        for neighbor in neighbors:
            if neighbor not in docno_to_index:
                raise KeyError(f"Neighbor {neighbor} of document {doc} not found in docno_to_index mapping.")
            
            neighbor_index = docno_to_index[neighbor]

            # Mark the connection in the matrix, ensuring symmetry
            adjacency_matrix[doc_index, neighbor_index] = adjacency_matrix[neighbor_index, doc_index] = 1

    return adjacency_matrix
    
    
def adjacency_matrix_to_coo(adjacency_matrix: np.ndarray) -> torch.Tensor:
    """
    Converts an adjacency matrix to COO format using PyTorch Geometric.

    Parameters:
    adjacency_matrix (np.ndarray): The adjacency matrix to be converted.

    Returns:
    torch.Tensor: Edge index tensor in COO format.
    """
    # Convert the numpy adjacency matrix to a SciPy sparse matrix (COO format)
    scipy_sparse_matrix = scipy.sparse.coo_matrix(adjacency_matrix)

    # Convert the SciPy sparse matrix to PyTorch Geometric COO format
    edge_index, edge_weight = from_scipy_sparse_matrix(scipy_sparse_matrix)

    return edge_index

In [ ]:
class GNRR(pt.Transformer):
    """
    A transformer that implements the Graph-based Adaptive Re-ranker algorithm from
    MacAvaney et al. "Adaptive Re-Ranking with a Corpus Graph" CIKM 2022.

    Required input columns: ['qid', 'query', 'docno', 'score', 'rank']
    Output columns: ['qid', 'query', 'docno', 'score', 'rank', 'iteration']
    where iteration defines the batch number which identified the document. Specifically
    even=initial retrieval   odd=corpus graph    -1=backfilled
    
    """
    def __init__(self,
        scorer: pt.Transformer,
        corpus_graph: 'CorpusGraph',
        flex_index,
        text_field = 'abstract'):
        self.scorer = scorer
        self.corpus_graph = corpus_graph
        self.flex_index = flex_index
        self.text_field = text_field

        self.dataset_retr = pt.get_dataset('irds:msmarco-passage')
        
       

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Applies Graph-based Adaptive Re-ranking to the provided dataframe. Essentially,
        Algorithm 1 from the paper.
        """
        result = {'qid': [], 'query': [], 'docno': [], 'rank': [], 'score': []}

        result = pd.DataFrame(result)
        ordered_queries = list(df.qid.unique())
        df = dict(iter(df.groupby(by=['qid'])))

        qids = df.keys()
            
        
        # RANKING ALGORITHM TO INCREASE THE RECALL
        
        payload = self.flex_index.payload()

        # RE-RANKING
        for i, qid in enumerate(ordered_queries):
            print(f"Currently processing: {i+1}/{len(qids)}")
            
            batch = df[qid].loc[:, ['qid', 'query', 'docno', 'score']]
                # go score the batch of document with the re-ranker

            add_texts = pt.text.get_text(self.dataset_retr, 'text')
            
            batch = add_texts(batch)

            # print(batch)
            inter_result = self.scorer.transform(batch, self.corpus_graph, corpus_graph_payload = payload)
            

            result = pd.concat([result, inter_result], axis=0, ignore_index=True)
       

            result['rank'] = result['rank'].astype(int)

        return result

In [ ]:
class Config:
    def __init__(self, modality, conv_type, hidden_dim, dropout_prob, n_layers, aggr, model_path, heads = 1, n_layers_mlp = 0, score = True, load = '', aggr_sage = 'mean'):
        self.modality = modality
        self.conv_type = conv_type
        self.hidden_dim = hidden_dim
        self.dropout_prob = dropout_prob
        self.n_layers = n_layers
        self.aggr = aggr
        self.heads = heads
        self.n_layers_mlp = n_layers_mlp
        self.model_path = model_path
        self.load = load
        self.score = score
        self.aggr_sage = aggr_sage



# config_local_gat_prev = Config(modality='local', conv_type='gat', hidden_dim=64, dropout_prob=0, score = True,
#                  n_layers=1, n_layers_mlp = 1, aggr='hadamard', model_path=f'./models/{dataset_id}/gat_8_True.pt', heads = 1)
config_local_gcn = Config(modality='multistage', conv_type='gcn', hidden_dim=128, dropout_prob=0.6, score = True,
                 n_layers=2, n_layers_mlp = 1, aggr='hadamard', model_path=f'models/msmarco_data/hadamard_2_789_0.01_0_128_0.6_gcn_multistage_1_tctcolbert.pt', heads = 1)


In [ ]:
dataset = pt.get_dataset("irds:msmarco-passage/train/split200-train")



In [ ]:
filtered_get_topics = dataset.get_topics()[15500:16000]

filtered_get_qrels = dataset.get_qrels()[dataset.get_qrels()['qid'].isin(filtered_get_topics['qid'])]
tct_scorer_local_gcn = GNRR_Scorer(config = config_local_gcn, text_field=text_field, get_data = True, qrels = filtered_get_qrels, dataset = 'msmarco_data')


In [ ]:
from pyterrier.measures import * 
pt.Experiment(
  [ 
    bm25 >> GNRR(tct_scorer_local_gcn, graph, flex_index, text_field=text_field)
  ],
  filtered_get_topics,
  filtered_get_qrels,
  names=['GCN'],#, 'MLP', 'MLP + GCN', 'MLP + GIN'],#, 'MLP', 'monoT5'],#, "TCTColbert+GCN", "TCTColbert+MLP+GCN", "TCTColbert+MLP", "MonoT5"],
  eval_metrics=[nDCG@10, P(rel = 2)@3, AP(rel=2), RR(rel = 2), R(rel=2)@1000] #[nDCG@10, nDCG@20, P(rel = 2)@10, P(rel = 2)@20, R(rel=2)@1000]
)

In [ ]:
dataset2 = pt.get_dataset("irds:msmarco-passage/train/split200-valid")
qrels_ = dataset2.get_qrels()
qrels = pd.concat([filtered_get_qrels, qrels_])

In [ ]:
qrels = qrels[['qid', 'iteration', 'docno', 'label']]

qrels.qid = qrels.qid.astype(str)
qrels.docno = qrels.docno.astype(str)
qrels.label = qrels.label.astype(int)
qrels.iteration = qrels.iteration.astype(int)

qrels.to_csv('data/msmarco_data/msmarco_data_qrels/qrels_val.txt', sep = ' ', index = False, header = False)
